# Issue Triage Agent: Classify, Label, Prioritize & Draft Responses

**Goal:** build an agent that ingests raw issue reports (bug reports, feature requests, support tickets), classifies them by type and component, suggests labels, estimates severity, and drafts an initial response — all using **structured outputs**.

---

## Why Issue Triage?

Open-source projects and support teams are flooded with issues. Manual triage is slow and inconsistent. An AI triage agent can:

1. **Classify** — bug, feature request, question, documentation, performance
2. **Label** — attach component/area labels automatically
3. **Prioritize** — estimate severity from P0 (critical) to P4 (cosmetic)
4. **Respond** — draft a templated first response acknowledging the issue
5. **Structure** — output every decision as a validated, machine-readable object

## Structured Outputs — The Key Concept

Most LLM outputs are free-form text. **Structured outputs** force the model (or our rule-based pipeline) to return data in a pre-defined schema — JSON, dataclass, or Pydantic model — so downstream code can consume it without fragile parsing.

### Why Structured Outputs Matter

| Problem with free text | How structured outputs fix it |
|---|---|
| Unpredictable format | Schema-validated JSON |
| Partial answers | Required fields enforced |
| Inconsistent labels | Enum-constrained values |
| Hard to test | Programmatic assertions on fields |
| Difficult to log/audit | Every decision is a typed record |

```text
Agent Pipeline

  Raw Issue Text
       │
       ▼
  ┌───────────────┐
  │  Preprocessor  │  → clean, tokenize, extract metadata
  └──────┬────────┘
         │
         ▼
  ┌───────────────┐
  │  Classifier    │  → issue_type, component, confidence
  └──────┬────────┘
         │
         ▼
  ┌───────────────┐
  │  Labeler       │  → suggested labels (multi-label)
  └──────┬────────┘
         │
         ▼
  ┌───────────────┐
  │  Severity Est. │  → P0–P4 + reasoning
  └──────┬────────┘
         │
         ▼
  ┌───────────────┐
  │  Responder     │  → draft response text
  └──────┬────────┘
         │
         ▼
  ┌───────────────┐
  │  Structured    │  → TriageResult (validated dataclass)
  │  Output        │
  └───────────────┘
```


## 1. Environment Setup

In [ ]:
!pip install -q pandas numpy matplotlib seaborn

In [ ]:
import json
import re
import textwrap
from collections import Counter, defaultdict
from dataclasses import asdict, dataclass, field
from datetime import datetime
from enum import Enum
from pathlib import Path
from typing import Any, Optional

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

SEED = 42
np.random.seed(SEED)

PROJECT_DIR = Path.cwd()
ARTIFACT_DIR = PROJECT_DIR / "artifacts"
ARTIFACT_DIR.mkdir(exist_ok=True)

print(f"Project dir:  {PROJECT_DIR}")
print(f"Artifact dir: {ARTIFACT_DIR}")

## 2. Defining Structured Output Schemas

**Structured outputs** start with a schema. We define every possible classification, label, and severity as an **Enum**, and every output as a **dataclass**. This guarantees:

- Every field has a defined type
- Enum fields can only take valid values
- Missing fields are caught at construction time
- The entire output is serializable to JSON

### Schema Design Principles

1. **Use Enums for categorical fields** — prevents typos and invalid values
2. **Use dataclasses for structured records** — typed, serializable, testable
3. **Include confidence scores** — the agent should express uncertainty
4. **Include reasoning** — every decision should be explainable
5. **Validate at construction** — catch errors immediately, not downstream


In [ ]:
# ── Issue Type ──
class IssueType(Enum):
    BUG = "bug"
    FEATURE_REQUEST = "feature_request"
    QUESTION = "question"
    DOCUMENTATION = "documentation"
    PERFORMANCE = "performance"
    SECURITY = "security"
    ENHANCEMENT = "enhancement"

    def __str__(self):
        return self.value


# ── Component ──
class Component(Enum):
    FRONTEND = "frontend"
    BACKEND = "backend"
    API = "api"
    DATABASE = "database"
    AUTH = "authentication"
    UI = "ui"
    CLI = "cli"
    DOCS = "docs"
    INFRA = "infrastructure"
    TESTING = "testing"
    UNKNOWN = "unknown"

    def __str__(self):
        return self.value


# ── Severity (P0 = critical, P4 = cosmetic) ──
class Severity(Enum):
    P0_CRITICAL = "P0"    # System down, data loss, security breach
    P1_HIGH = "P1"        # Major feature broken, no workaround
    P2_MEDIUM = "P2"      # Feature degraded, workaround exists
    P3_LOW = "P3"         # Minor inconvenience
    P4_COSMETIC = "P4"    # Visual/typo, no functional impact

    def __str__(self):
        return self.value

    @property
    def description(self):
        return {
            "P0": "Critical — system down, data loss, or security breach",
            "P1": "High — major feature broken, no workaround",
            "P2": "Medium — feature degraded, workaround exists",
            "P3": "Low — minor inconvenience",
            "P4": "Cosmetic — visual/typo, no functional impact",
        }[self.value]


# ── Structured output dataclasses ──
@dataclass
class Classification:
    issue_type: IssueType
    confidence: float          # 0.0 to 1.0
    reasoning: str

    def validate(self) -> list[str]:
        errors = []
        if not 0.0 <= self.confidence <= 1.0:
            errors.append(f"confidence must be 0-1, got {self.confidence}")
        if not isinstance(self.issue_type, IssueType):
            errors.append(f"issue_type must be IssueType, got {type(self.issue_type)}")
        return errors


@dataclass
class ComponentMatch:
    component: Component
    confidence: float
    evidence: str              # What text triggered this match


@dataclass
class LabelSuggestion:
    labels: list[str]
    component_matches: list[ComponentMatch]
    auto_apply: list[str]      # High-confidence labels to apply automatically
    needs_review: list[str]    # Low-confidence labels needing human review


@dataclass
class SeverityEstimate:
    severity: Severity
    confidence: float
    reasoning: str
    signals: list[str]         # What signals drove the estimate
    escalate: bool             # Should this be escalated to on-call?


@dataclass
class DraftResponse:
    greeting: str
    acknowledgment: str
    next_steps: list[str]
    closing: str
    full_text: str
    tone: str                  # "empathetic", "technical", "brief"


@dataclass
class TriageResult:
    """The complete structured output for a triaged issue."""
    issue_id: str
    title: str
    classification: Classification
    labels: LabelSuggestion
    severity: SeverityEstimate
    response: DraftResponse
    triaged_at: str = ""
    agent_version: str = "1.0.0"

    def __post_init__(self):
        if not self.triaged_at:
            self.triaged_at = datetime.now().isoformat()

    def validate(self) -> list[str]:
        """Validate all fields. Returns list of errors (empty = valid)."""
        errors = self.classification.validate()
        if not self.labels.labels:
            errors.append("At least one label is required")
        if not 0.0 <= self.severity.confidence <= 1.0:
            errors.append(f"severity confidence must be 0-1, got {self.severity.confidence}")
        if not self.response.full_text:
            errors.append("Response text is required")
        return errors

    def to_dict(self) -> dict:
        """Serialize to a JSON-compatible dictionary."""
        d = asdict(self)
        d["classification"]["issue_type"] = str(self.classification.issue_type)
        d["severity"]["severity"] = str(self.severity.severity)
        for cm in d["labels"]["component_matches"]:
            cm["component"] = str(cm["component"]) if isinstance(cm.get("component"), Component) else cm["component"]
        return d

    def to_json(self, indent: int = 2) -> str:
        return json.dumps(self.to_dict(), indent=indent)


# Preview the schema
print("Structured Output Schema:")
print(f"  IssueType options:  {[e.value for e in IssueType]}")
print(f"  Component options:  {[e.value for e in Component]}")
print(f"  Severity options:   {[e.value for e in Severity]}")
print(f"\nTriageResult fields:")
for f_name, f_type in TriageResult.__dataclass_fields__.items():
    print(f"  {f_name:20s} → {f_type.type.__name__ if hasattr(f_type.type, '__name__') else f_type.type}")

## 3. Synthetic Issue Dataset

We create realistic issue reports that cover the full spectrum of types, severities, and components.


In [ ]:
ISSUES = [
    {
        "id": "ISS-001",
        "title": "Login page returns 500 error after latest deploy",
        "body": (
            "After the v2.3.1 deploy this morning, users cannot log in. "
            "The login page shows a 500 Internal Server Error. This affects "
            "all users including admins. We're getting reports from over 200 "
            "users. The error log shows a null pointer exception in the auth "
            "module. No workaround exists — users are completely locked out."
        ),
        "reporter": "ops-team",
        "created": "2024-06-15T09:23:00Z",
    },
    {
        "id": "ISS-002",
        "title": "Add dark mode support",
        "body": (
            "It would be great to have a dark mode option in the settings. "
            "Many modern apps support this and it reduces eye strain. "
            "Could we add a toggle in the user preferences panel? "
            "This is not urgent but would be a nice improvement."
        ),
        "reporter": "user-jane",
        "created": "2024-06-14T14:00:00Z",
    },
    {
        "id": "ISS-003",
        "title": "How do I reset my password?",
        "body": (
            "I forgot my password and can't figure out how to reset it. "
            "I don't see a 'Forgot Password' link on the login page. "
            "Can someone help me or point me to the right documentation?"
        ),
        "reporter": "user-bob",
        "created": "2024-06-14T11:30:00Z",
    },
    {
        "id": "ISS-004",
        "title": "API response time degraded 10x since database migration",
        "body": (
            "Since we migrated to the new PostgreSQL cluster on June 10, "
            "the /api/v2/search endpoint response time went from 50ms to "
            "500ms average. The p99 latency is now 2.5 seconds. This is "
            "causing timeouts for mobile clients. Query execution plans "
            "show a missing index on the search_history table. Affects "
            "approximately 30,000 API calls per hour."
        ),
        "reporter": "backend-team",
        "created": "2024-06-13T16:45:00Z",
    },
    {
        "id": "ISS-005",
        "title": "Typo in API docs: 'authetication' should be 'authentication'",
        "body": (
            "On the API reference page under the Authentication section, "
            "the word 'authentication' is misspelled as 'authetication' "
            "in three places. Also the code example shows the old v1 "
            "endpoint instead of v2."
        ),
        "reporter": "contributor-alex",
        "created": "2024-06-13T10:00:00Z",
    },
    {
        "id": "ISS-006",
        "title": "SQL injection vulnerability in search endpoint",
        "body": (
            "I found a SQL injection vulnerability in the /search endpoint. "
            "By passing a crafted query parameter, it's possible to extract "
            "data from other tables. I have a proof-of-concept but won't "
            "share it publicly. Steps to reproduce: send a GET request to "
            "/api/search?q=' OR '1'='1 and observe the response includes "
            "data from the users table. This is a critical security issue."
        ),
        "reporter": "security-researcher",
        "created": "2024-06-15T08:00:00Z",
    },
    {
        "id": "ISS-007",
        "title": "Dashboard chart tooltip shows wrong currency",
        "body": (
            "On the revenue dashboard, the bar chart tooltip shows values "
            "in USD but our account is set to EUR. The chart itself shows "
            "correct EUR values. Only the tooltip is wrong. This is "
            "cosmetic — the actual data is correct."
        ),
        "reporter": "user-maria",
        "created": "2024-06-12T09:15:00Z",
    },
    {
        "id": "ISS-008",
        "title": "CLI tool crashes with large file input",
        "body": (
            "When running 'mytool process --input bigfile.csv' with a CSV "
            "file larger than 2GB, the CLI crashes with an OutOfMemoryError. "
            "The tool works fine with files under 500MB. There should be "
            "either streaming support or a clear error message with the "
            "file size limit. Stack trace attached."
        ),
        "reporter": "user-dev-k",
        "created": "2024-06-11T17:30:00Z",
    },
    {
        "id": "ISS-009",
        "title": "Improve onboarding flow with guided tour",
        "body": (
            "New users often get confused after signup. We should add an "
            "interactive guided tour that walks them through: 1) creating "
            "their first project, 2) inviting team members, 3) setting up "
            "notifications. Similar to what Notion and Linear do. This would "
            "reduce support tickets from new users by an estimated 40%."
        ),
        "reporter": "pm-sarah",
        "created": "2024-06-10T13:00:00Z",
    },
    {
        "id": "ISS-010",
        "title": "Database connection pool exhaustion under load",
        "body": (
            "During peak traffic (>5000 req/s), the database connection pool "
            "is exhausted and requests start queuing. The pool is configured "
            "for 50 connections but the application is opening connections "
            "without properly returning them. This causes cascading failures "
            "across all services. We need connection pooling with PgBouncer "
            "and proper connection lifecycle management."
        ),
        "reporter": "sre-team",
        "created": "2024-06-15T07:00:00Z",
    },
    {
        "id": "ISS-011",
        "title": "Export button does nothing on Firefox",
        "body": (
            "The 'Export to CSV' button on the reports page doesn't do "
            "anything when clicked in Firefox 126. It works in Chrome. "
            "No errors in the console. Probably a browser compatibility "
            "issue with the download API."
        ),
        "reporter": "user-chen",
        "created": "2024-06-14T16:00:00Z",
    },
    {
        "id": "ISS-012",
        "title": "Add webhook support for CI/CD integration",
        "body": (
            "We'd like to trigger CI/CD pipeline runs via webhooks when "
            "certain events occur (e.g., deployment completed, test suite "
            "finished). Please add a webhook configuration page in project "
            "settings with support for custom headers and payload templates. "
            "This is a common feature in competing products."
        ),
        "reporter": "team-devops",
        "created": "2024-06-09T11:00:00Z",
    },
]

print(f"Created {len(ISSUES)} synthetic issues:\n")
for iss in ISSUES:
    print(f"  {iss['id']}  {iss['title'][:60]}")

## 4. Issue Classifier

The classifier determines the **type** of issue using keyword matching and pattern analysis. In production, an LLM or fine-tuned model replaces keyword rules, but the **output schema** stays the same.


In [ ]:
# Keyword groups for classification
TYPE_KEYWORDS: dict[IssueType, list[str]] = {
    IssueType.BUG: [
        r"\berror\b", r"\bbug\b", r"\bcrash", r"\bfail", r"\bbroken\b",
        r"\b500\b", r"\b404\b", r"\bexception\b", r"\bnot working\b",
        r"\bdoesn.t work\b", r"\bdoes nothing\b", r"\bnull pointer\b",
    ],
    IssueType.FEATURE_REQUEST: [
        r"\badd\b.*\bsupport\b", r"\bfeature\b", r"\bwould be great\b",
        r"\bplease add\b", r"\bwe.d like\b", r"\bshould add\b",
        r"\bimprove\b.*\bflow\b", r"\benhancement\b",
    ],
    IssueType.QUESTION: [
        r"\bhow do i\b", r"\bhow to\b", r"\bcan someone help\b",
        r"\bwhere (?:can|do) i\b", r"\bpoint me to\b",
        r"\bwhat is\b", r"\bis there a way\b",
    ],
    IssueType.DOCUMENTATION: [
        r"\btypo\b", r"\bdocs?\b", r"\bdocumentation\b", r"\bmisspell",
        r"\bcode example\b", r"\breference page\b", r"\bREADME\b",
    ],
    IssueType.PERFORMANCE: [
        r"\bslow\b", r"\blatency\b", r"\bresponse time\b", r"\btimeout\b",
        r"\bperformance\b", r"\bdegraded\b", r"\bp99\b", r"\boutofmemory\b",
        r"\bconnection pool\b", r"\bexhaust",
    ],
    IssueType.SECURITY: [
        r"\bsecurity\b", r"\bvulnerability\b", r"\binjection\b", r"\bxss\b",
        r"\bcve\b", r"\bexploit\b", r"\bbreach\b", r"\bunauthori[sz]ed\b",
    ],
    IssueType.ENHANCEMENT: [
        r"\bimprove\b", r"\bbetter\b", r"\boptimi[sz]e\b",
        r"\brefactor\b", r"\bwould be nice\b",
    ],
}


def classify_issue(title: str, body: str) -> Classification:
    """Classify an issue by type using keyword matching."""
    text = f"{title} {body}".lower()

    scores: dict[IssueType, float] = {}

    for issue_type, patterns in TYPE_KEYWORDS.items():
        matches = sum(1 for p in patterns if re.search(p, text, re.IGNORECASE))
        total = len(patterns)
        scores[issue_type] = matches / total if total > 0 else 0

    # Pick the best match
    if not scores or max(scores.values()) == 0:
        return Classification(
            issue_type=IssueType.BUG,
            confidence=0.1,
            reasoning="No keywords matched; defaulting to bug",
        )

    best_type = max(scores, key=scores.get)
    best_score = scores[best_type]

    # If security and bug both match, security takes priority
    if scores.get(IssueType.SECURITY, 0) > 0.1:
        best_type = IssueType.SECURITY
        best_score = max(best_score, scores[IssueType.SECURITY])

    # Determine reasoning by listing matched patterns
    matched = [p for p in TYPE_KEYWORDS[best_type]
               if re.search(p, text, re.IGNORECASE)]
    reasoning = f"Matched {len(matched)} keyword(s) for {best_type.value}: " + \
                ", ".join(m.replace(r"\b", "").replace("\\", "") for m in matched[:3])

    confidence = min(best_score * 1.5, 0.99)  # Scale up but cap at 0.99

    return Classification(
        issue_type=best_type,
        confidence=round(confidence, 2),
        reasoning=reasoning,
    )


# Test on all issues
print(f"{'ID':<10} {'Classified As':<20} {'Confidence':<12} {'Reasoning'}")
print("-" * 85)
for iss in ISSUES:
    clf = classify_issue(iss["title"], iss["body"])
    print(f"{iss['id']:<10} {str(clf.issue_type):<20} {clf.confidence:<12.2f} {clf.reasoning[:50]}")

## 5. Component Detection & Label Suggestion

The labeler identifies which **component** or **area** of the system the issue relates to, and suggests labels that could be applied in a project management tool (GitHub Issues, Jira, Linear).


In [ ]:
COMPONENT_KEYWORDS: dict[Component, list[str]] = {
    Component.FRONTEND: [
        r"\bui\b", r"\bfrontend\b", r"\bbutton\b", r"\bpage\b",
        r"\bdashboard\b", r"\bchart\b", r"\btooltip\b", r"\breact\b",
        r"\bbrowser\b", r"\bfirefox\b", r"\bchrome\b", r"\bcss\b",
    ],
    Component.BACKEND: [
        r"\bbackend\b", r"\bserver\b", r"\bendpoint\b",
        r"\bservice\b", r"\b500\b", r"\bmodule\b",
    ],
    Component.API: [
        r"\bapi\b", r"\bendpoint\b", r"\brest\b", r"\brequest\b",
        r"\bresponse\b", r"\bget\b.*\brequest\b", r"\bwebhook\b",
    ],
    Component.DATABASE: [
        r"\bdatabase\b", r"\bsql\b", r"\bquery\b", r"\bindex\b",
        r"\bpostgresql\b", r"\bconnection pool\b", r"\bmigrat",
    ],
    Component.AUTH: [
        r"\blogin\b", r"\bauth", r"\bpassword\b", r"\bsignup\b",
        r"\bsession\b", r"\btoken\b", r"\bpermission\b",
    ],
    Component.UI: [
        r"\bui\b", r"\bux\b", r"\bdesign\b", r"\blayout\b",
        r"\bdark mode\b", r"\btheme\b", r"\btooltip\b", r"\bexport\b",
    ],
    Component.CLI: [
        r"\bcli\b", r"\bcommand.line\b", r"\btool\b.*\bprocess\b",
        r"\bmytool\b", r"\bterminal\b",
    ],
    Component.DOCS: [
        r"\bdocs?\b", r"\bdocumentation\b", r"\breadme\b",
        r"\bexample\b", r"\breference\b",
    ],
    Component.INFRA: [
        r"\bdeploy\b", r"\bci.?cd\b", r"\binfra\b", r"\bcluster\b",
        r"\bpgbouncer\b", r"\bload\b", r"\bscaling\b",
    ],
    Component.TESTING: [
        r"\btest\b", r"\bunit test\b", r"\be2e\b", r"\bcoverage\b",
    ],
}


LABEL_MAP = {
    IssueType.BUG: ["bug", "needs-investigation"],
    IssueType.FEATURE_REQUEST: ["enhancement", "needs-design"],
    IssueType.QUESTION: ["question", "help-wanted"],
    IssueType.DOCUMENTATION: ["documentation", "good-first-issue"],
    IssueType.PERFORMANCE: ["performance", "needs-profiling"],
    IssueType.SECURITY: ["security", "priority-critical"],
    IssueType.ENHANCEMENT: ["enhancement"],
}


def detect_components(title: str, body: str) -> list[ComponentMatch]:
    """Detect which components the issue relates to."""
    text = f"{title} {body}".lower()
    matches = []

    for comp, patterns in COMPONENT_KEYWORDS.items():
        hits = [p for p in patterns if re.search(p, text, re.IGNORECASE)]
        if hits:
            conf = min(len(hits) / len(patterns) * 1.5, 0.99)
            evidence = ", ".join(
                re.search(p, text, re.IGNORECASE).group()
                for p in hits[:3]
                if re.search(p, text, re.IGNORECASE)
            )
            matches.append(ComponentMatch(
                component=comp,
                confidence=round(conf, 2),
                evidence=evidence,
            ))

    matches.sort(key=lambda m: m.confidence, reverse=True)
    return matches


def suggest_labels(classification: Classification,
                   components: list[ComponentMatch]) -> LabelSuggestion:
    """Suggest labels based on classification and detected components."""
    labels = list(LABEL_MAP.get(classification.issue_type, []))

    # Add component labels
    for cm in components:
        label = f"area:{cm.component.value}"
        if label not in labels:
            labels.append(label)

    # Split into auto-apply (high confidence) and needs-review (low confidence)
    auto_apply = []
    needs_review = []

    for label in labels:
        # Type-based labels auto-apply if classification confidence is high
        if label in LABEL_MAP.get(classification.issue_type, []):
            if classification.confidence >= 0.5:
                auto_apply.append(label)
            else:
                needs_review.append(label)
        # Component labels auto-apply if component confidence is high
        elif label.startswith("area:"):
            comp_name = label.split(":")[1]
            matching = [cm for cm in components if cm.component.value == comp_name]
            if matching and matching[0].confidence >= 0.5:
                auto_apply.append(label)
            else:
                needs_review.append(label)
        else:
            needs_review.append(label)

    return LabelSuggestion(
        labels=labels,
        component_matches=components,
        auto_apply=auto_apply,
        needs_review=needs_review,
    )


# Test
for iss in ISSUES[:4]:
    clf = classify_issue(iss["title"], iss["body"])
    comps = detect_components(iss["title"], iss["body"])
    labels = suggest_labels(clf, comps)
    print(f"{iss['id']}: {iss['title'][:45]}")
    print(f"  Components: {[str(c.component) for c in comps[:3]]}")
    print(f"  Labels:     {labels.labels}")
    print(f"  Auto-apply: {labels.auto_apply}")
    print(f"  Review:     {labels.needs_review}")
    print()

## 6. Severity Estimation

Severity is estimated from multiple **signals** in the issue text:
- Impact scope (all users vs one user)
- Workaround availability
- Data loss risk
- Security implications
- Urgency language
- Numbers (affected users, latency, etc.)


In [ ]:
SEVERITY_SIGNALS = {
    "all_users_affected": {
        "patterns": [r"\ball users\b", r"\beveryone\b", r"\ball .* affected\b", r"\bcompletely\b"],
        "weight": 2.0,
        "direction": "up",
    },
    "no_workaround": {
        "patterns": [r"\bno workaround\b", r"\bcompletely locked\b", r"\bcannot\b.*\bat all\b"],
        "weight": 2.0,
        "direction": "up",
    },
    "data_loss": {
        "patterns": [r"\bdata loss\b", r"\bcorrupt", r"\bdelete.*all\b"],
        "weight": 3.0,
        "direction": "up",
    },
    "security": {
        "patterns": [r"\bsecurity\b", r"\bvulnerability\b", r"\binjection\b", r"\bexploit\b",
                     r"\bbreach\b"],
        "weight": 3.0,
        "direction": "up",
    },
    "high_traffic": {
        "patterns": [r"\d{3,}\s*(?:users?|requests?|calls?)\b", r"\bpeak\b.*\btraffic\b"],
        "weight": 1.5,
        "direction": "up",
    },
    "cascading_failure": {
        "patterns": [r"\bcascading\b", r"\ball services\b", r"\bsystem down\b"],
        "weight": 2.5,
        "direction": "up",
    },
    "workaround_exists": {
        "patterns": [r"\bworkaround\b", r"\bworks? (?:fine )??in\b", r"\bcosemtic\b|cosmetic"],
        "weight": -1.0,
        "direction": "down",
    },
    "not_urgent": {
        "patterns": [r"\bnot urgent\b", r"\bnice (?:to have|improvement)\b",
                     r"\bwould be (?:great|nice)\b", r"\bnon.?critical\b"],
        "weight": -1.5,
        "direction": "down",
    },
    "cosmetic": {
        "patterns": [r"\btypo\b", r"\bcosmetic\b", r"\bvisual\b", r"\bmisspell"],
        "weight": -2.0,
        "direction": "down",
    },
}


def estimate_severity(title: str, body: str,
                      classification: Classification) -> SeverityEstimate:
    """Estimate severity of an issue using signal analysis."""
    text = f"{title} {body}".lower()
    matched_signals = []
    score = 5.0  # Start at midpoint

    for signal_name, config in SEVERITY_SIGNALS.items():
        for pattern in config["patterns"]:
            if re.search(pattern, text, re.IGNORECASE):
                matched_signals.append(signal_name)
                score += config["weight"]
                break  # One match per signal group

    # Security issues are always at least P1
    if classification.issue_type == IssueType.SECURITY:
        score = max(score, 9.0)

    # Questions are always P3 or P4
    if classification.issue_type == IssueType.QUESTION:
        score = min(score, 3.0)

    # Documentation issues cap at P3
    if classification.issue_type == IssueType.DOCUMENTATION:
        score = min(score, 4.0)

    # Map score to severity
    if score >= 9:
        severity = Severity.P0_CRITICAL
    elif score >= 7:
        severity = Severity.P1_HIGH
    elif score >= 5:
        severity = Severity.P2_MEDIUM
    elif score >= 3:
        severity = Severity.P3_LOW
    else:
        severity = Severity.P4_COSMETIC

    # Confidence based on signal count
    confidence = min(0.5 + len(matched_signals) * 0.12, 0.95)

    # Escalation trigger
    escalate = severity in (Severity.P0_CRITICAL, Severity.P1_HIGH) and \
               classification.issue_type in (IssueType.SECURITY, IssueType.BUG)

    reasoning = (
        f"Score: {score:.1f}/14. "
        f"Matched {len(matched_signals)} signal(s). "
        f"{severity.description}."
    )

    return SeverityEstimate(
        severity=severity,
        confidence=round(confidence, 2),
        reasoning=reasoning,
        signals=matched_signals,
        escalate=escalate,
    )


# Test on all issues
print(f"{'ID':<10} {'Severity':<6} {'Conf':<6} {'Escalate':<10} {'Signals'}")
print("-" * 80)
for iss in ISSUES:
    clf = classify_issue(iss["title"], iss["body"])
    sev = estimate_severity(iss["title"], iss["body"], clf)
    print(f"{iss['id']:<10} {str(sev.severity):<6} {sev.confidence:<6.2f} "
          f"{'YES' if sev.escalate else 'no':<10} {', '.join(sev.signals[:4])}")

## 7. Response Drafter

The responder generates a **structured draft response** appropriate to the issue type and severity.


In [ ]:
RESPONSE_TEMPLATES = {
    IssueType.BUG: {
        "greeting": "Thank you for reporting this issue.",
        "ack": "We've confirmed this is a bug and have added it to our active triage queue.",
        "steps": [
            "Our engineering team is investigating the root cause.",
            "We'll provide an update within {sla} hours.",
            "In the meantime, {workaround}.",
        ],
        "closing": "We'll keep this issue updated with our progress.",
        "tone": "empathetic",
    },
    IssueType.FEATURE_REQUEST: {
        "greeting": "Thanks for the feature suggestion!",
        "ack": "We've logged this as a feature request for review by our product team.",
        "steps": [
            "The product team will evaluate this in the next planning cycle.",
            "We'll update the priority based on user demand and feasibility.",
            "Feel free to add more context or use cases.",
        ],
        "closing": "We appreciate your input in making the product better.",
        "tone": "brief",
    },
    IssueType.QUESTION: {
        "greeting": "Thanks for reaching out!",
        "ack": "This is a great question.",
        "steps": [
            "{answer_hint}",
            "You can also check our documentation at docs.example.com.",
            "If that doesn't help, let us know and we'll dig deeper.",
        ],
        "closing": "Don't hesitate to ask if you have more questions!",
        "tone": "empathetic",
    },
    IssueType.DOCUMENTATION: {
        "greeting": "Thanks for catching this!",
        "ack": "We've confirmed the documentation issue.",
        "steps": [
            "A fix has been queued for the docs.",
            "If you'd like to contribute, PRs are welcome!",
        ],
        "closing": "Thanks for helping improve our docs.",
        "tone": "brief",
    },
    IssueType.PERFORMANCE: {
        "greeting": "Thank you for the detailed performance report.",
        "ack": "We're treating this as a performance regression with high priority.",
        "steps": [
            "Our SRE team is analyzing the metrics you provided.",
            "We'll investigate the identified bottleneck.",
            "An update will follow within {sla} hours.",
        ],
        "closing": "We take performance seriously and will resolve this promptly.",
        "tone": "technical",
    },
    IssueType.SECURITY: {
        "greeting": "Thank you for the responsible disclosure.",
        "ack": "We're treating this as a critical security issue.",
        "steps": [
            "Our security team has been notified immediately.",
            "We will not discuss details publicly until a fix is deployed.",
            "We'll reach out privately to coordinate the fix and timeline.",
            "A CVE will be issued if applicable.",
        ],
        "closing": "We appreciate your responsible approach to reporting this.",
        "tone": "technical",
    },
    IssueType.ENHANCEMENT: {
        "greeting": "Thanks for the enhancement suggestion!",
        "ack": "We've noted this for consideration.",
        "steps": [
            "We'll evaluate this against our current roadmap.",
            "Community feedback helps us prioritize — upvotes are welcome.",
        ],
        "closing": "Thanks for contributing to the project!",
        "tone": "brief",
    },
}


SLA_MAP = {
    Severity.P0_CRITICAL: "4",
    Severity.P1_HIGH: "12",
    Severity.P2_MEDIUM: "48",
    Severity.P3_LOW: "1 week",
    Severity.P4_COSMETIC: "next release",
}


def draft_response(classification: Classification,
                   severity: SeverityEstimate,
                   title: str, body: str) -> DraftResponse:
    """Draft a structured response appropriate to the issue."""
    template = RESPONSE_TEMPLATES.get(
        classification.issue_type,
        RESPONSE_TEMPLATES[IssueType.BUG]
    )

    sla = SLA_MAP.get(severity.severity, "48")

    # Generate contextual workaround suggestion
    workaround = "we are investigating workarounds"
    if "workaround" in body.lower():
        workaround = "we understand no workaround is currently available"

    # Generate answer hint for questions
    answer_hint = "We're looking into your question and will respond shortly."
    if "password" in body.lower():
        answer_hint = ("To reset your password, look for the 'Forgot Password' link "
                       "on the login page, or contact support@example.com.")

    steps = [
        s.format(sla=sla, workaround=workaround, answer_hint=answer_hint)
        for s in template["steps"]
    ]

    full_text = (
        f"{template['greeting']}\n\n"
        f"{template['ack']}\n\n"
        f"**Next steps:**\n" +
        "\n".join(f"- {s}" for s in steps) +
        f"\n\n{template['closing']}"
    )

    return DraftResponse(
        greeting=template["greeting"],
        acknowledgment=template["ack"],
        next_steps=steps,
        closing=template["closing"],
        full_text=full_text,
        tone=template["tone"],
    )


# Preview a few responses
for iss in [ISSUES[0], ISSUES[2], ISSUES[5]]:
    clf = classify_issue(iss["title"], iss["body"])
    sev = estimate_severity(iss["title"], iss["body"], clf)
    resp = draft_response(clf, sev, iss["title"], iss["body"])
    print(f"{'='*60}")
    print(f"{iss['id']}: {iss['title'][:50]}")
    print(f"Tone: {resp.tone}")
    print(f"{'='*60}")
    print(resp.full_text)
    print()

## 8. Assembling the Full Agent

In [ ]:
class IssuTriageAgent:
    """End-to-end issue triage agent producing structured outputs."""

    def __init__(self):
        self._results: list[TriageResult] = []

    def triage(self, issue: dict, verbose: bool = False) -> TriageResult:
        """Triage a single issue end to end."""
        title = issue["title"]
        body  = issue["body"]
        issue_id = issue.get("id", "UNKNOWN")

        # Step 1: Classify
        classification = classify_issue(title, body)
        if verbose:
            print(f"[1] Classification: {classification.issue_type} "
                  f"(conf={classification.confidence:.2f})")

        # Step 2: Detect components & suggest labels
        components = detect_components(title, body)
        labels = suggest_labels(classification, components)
        if verbose:
            print(f"[2] Labels: {labels.labels}")

        # Step 3: Estimate severity
        severity = estimate_severity(title, body, classification)
        if verbose:
            print(f"[3] Severity: {severity.severity} "
                  f"(conf={severity.confidence:.2f}, escalate={severity.escalate})")

        # Step 4: Draft response
        response = draft_response(classification, severity, title, body)
        if verbose:
            print(f"[4] Response drafted ({response.tone} tone, "
                  f"{len(response.next_steps)} steps)")

        # Step 5: Assemble structured output
        result = TriageResult(
            issue_id=issue_id,
            title=title,
            classification=classification,
            labels=labels,
            severity=severity,
            response=response,
        )

        # Step 6: Validate
        errors = result.validate()
        if errors:
            print(f"  VALIDATION ERRORS: {errors}")
        elif verbose:
            print(f"[5] Validated: OK")

        self._results.append(result)
        return result

    def triage_batch(self, issues: list[dict]) -> list[TriageResult]:
        """Triage multiple issues."""
        return [self.triage(iss) for iss in issues]

    @property
    def results(self):
        return self._results

    def summary_dataframe(self) -> pd.DataFrame:
        """Return all triage results as a DataFrame."""
        rows = []
        for r in self._results:
            rows.append({
                "issue_id": r.issue_id,
                "title": r.title[:40],
                "type": str(r.classification.issue_type),
                "type_conf": r.classification.confidence,
                "severity": str(r.severity.severity),
                "sev_conf": r.severity.confidence,
                "escalate": r.severity.escalate,
                "labels": ", ".join(r.labels.auto_apply),
                "review_labels": ", ".join(r.labels.needs_review),
                "tone": r.response.tone,
            })
        return pd.DataFrame(rows)


agent = IssuTriageAgent()
print("IssuTriageAgent ready.")

## 9. Running the Agent — Full Triage

In [ ]:
all_results = agent.triage_batch(ISSUES)

print(f"Triaged {len(all_results)} issues.\n")
df = agent.summary_dataframe()
print(df.to_string(index=False))

## 10. Verbose Walkthrough — Single Issue

In [ ]:
# Reset agent for clean walkthrough
walkthrough_agent = IssuTriageAgent()
print("VERBOSE TRIAGE WALKTHROUGH")
print("=" * 60)
print(f"Issue: {ISSUES[0]['title']}")
print(f"Body:  {ISSUES[0]['body'][:100]}...")
print("=" * 60)
print()

result = walkthrough_agent.triage(ISSUES[0], verbose=True)

print(f"\n{'='*60}")
print("COMPLETE STRUCTURED OUTPUT (JSON):")
print("=" * 60)
print(result.to_json(indent=2))

## 11. Structured Outputs Deep Dive

### What Makes an Output "Structured"?

A structured output is a **typed, validated, machine-readable** record. Compare:

**Unstructured** (free text):
```
This looks like a P0 bug in the auth module. We should add the "bug" and 
"authentication" labels. Someone should look at it urgently.
```

**Structured** (typed record):
```json
{
  "classification": {
    "issue_type": "bug",
    "confidence": 0.92,
    "reasoning": "Matched 4 keywords: error, 500, crash, null pointer"
  },
  "severity": {
    "severity": "P0",
    "confidence": 0.88,
    "escalate": true,
    "signals": ["all_users_affected", "no_workaround"]
  },
  "labels": {
    "auto_apply": ["bug", "needs-investigation", "area:authentication"],
    "needs_review": []
  }
}
```

### Why Structure Matters

| Perspective | Unstructured | Structured |
|---|---|---|
| **Downstream code** | Regex parsing, brittle | `result.severity.severity == Severity.P0` |
| **Validation** | Impossible | `result.validate()` → list of errors |
| **Logging** | Free text blobs | JSON → database → dashboards |
| **Testing** | Compare strings | Assert field values |
| **Aggregation** | Manual counting | `df.groupby("severity").count()` |
| **Audit trail** | Ambiguous | Every decision has a confidence + reasoning |

### Schema Design Patterns

#### 1. Use Enums for Categorical Fields
```python
class Severity(Enum):
    P0 = "P0"  # Prevents typos like "p0", "P-0", "critical"
```

#### 2. Include Confidence Scores
```python
@dataclass
class Classification:
    issue_type: IssueType
    confidence: float  # 0.0–1.0, enables threshold-based routing
```

#### 3. Separate Auto-Apply from Needs-Review
```python
@dataclass
class LabelSuggestion:
    auto_apply: list[str]   # High confidence → apply automatically
    needs_review: list[str] # Low confidence → human decides
```

#### 4. Add Reasoning to Every Decision
```python
@dataclass
class SeverityEstimate:
    severity: Severity
    reasoning: str  # "Score: 11.5/14. Matched 4 signals."
    signals: list[str]  # ["all_users_affected", "no_workaround"]
```

#### 5. Validate at Construction
```python
@dataclass
class TriageResult:
    def validate(self) -> list[str]:
        errors = []
        if not 0 <= self.severity.confidence <= 1:
            errors.append("Invalid confidence")
        return errors
```

### OpenAI Structured Outputs (Production)

```python
from pydantic import BaseModel
from openai import OpenAI

class TriageOutput(BaseModel):
    issue_type: str
    severity: str
    labels: list[str]
    response: str

client = OpenAI()
completion = client.beta.chat.completions.parse(
    model="gpt-4o",
    messages=[{"role": "user", "content": issue_text}],
    response_format=TriageOutput,  # Enforced JSON schema!
)
result: TriageOutput = completion.choices[0].message.parsed
```

This guarantees the LLM output conforms to the schema — no parsing needed.


## 12. Triage Analytics

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# Chart 1: Issue type distribution
ax = axes[0, 0]
type_counts = df["type"].value_counts()
ax.barh(type_counts.index, type_counts.values, color="#1f77b4", alpha=0.8)
ax.set_xlabel("Count")
ax.set_title("Issue Type Distribution")

# Chart 2: Severity distribution
ax2 = axes[0, 1]
sev_order = ["P0", "P1", "P2", "P3", "P4"]
sev_colors = {"P0": "#d62728", "P1": "#ff7f0e", "P2": "#f0c929",
              "P3": "#2ca02c", "P4": "#1f77b4"}
sev_counts = df["severity"].value_counts()
sev_items = [(s, sev_counts.get(s, 0)) for s in sev_order if s in sev_counts.index]
if sev_items:
    bars = ax2.bar([s[0] for s in sev_items], [s[1] for s in sev_items],
                   color=[sev_colors[s[0]] for s in sev_items], alpha=0.8)
    ax2.set_ylabel("Count")
    ax2.set_title("Severity Distribution")

# Chart 3: Confidence scores
ax3 = axes[1, 0]
ax3.scatter(df["type_conf"], df["sev_conf"], c=df["escalate"].map({True: "red", False: "blue"}),
            s=80, alpha=0.7, edgecolors="black", linewidth=0.5)
ax3.set_xlabel("Classification Confidence")
ax3.set_ylabel("Severity Confidence")
ax3.set_title("Confidence Scatter (red = escalate)")
ax3.axhline(y=0.5, color="gray", linestyle="--", alpha=0.3)
ax3.axvline(x=0.5, color="gray", linestyle="--", alpha=0.3)

# Chart 4: Label frequency
ax4 = axes[1, 1]
all_labels = []
for r in agent.results:
    all_labels.extend(r.labels.auto_apply)
    all_labels.extend(r.labels.needs_review)
label_counts = Counter(all_labels).most_common(10)
if label_counts:
    ax4.barh([l[0] for l in label_counts], [l[1] for l in label_counts],
             color="#9467bd", alpha=0.8)
    ax4.set_xlabel("Frequency")
    ax4.set_title("Top Labels Suggested")

plt.suptitle("Issue Triage Agent — Analytics", fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

## 13. Escalation Report

In [ ]:
escalated = [r for r in agent.results if r.severity.escalate]
non_escalated = [r for r in agent.results if not r.severity.escalate]

print("ESCALATION REPORT")
print("=" * 70)
print(f"\n🚨 ESCALATED ({len(escalated)} issues):")
for r in escalated:
    print(f"  [{r.severity.severity}] {r.issue_id}: {r.title[:50]}")
    print(f"       Type: {r.classification.issue_type} | "
          f"Signals: {', '.join(r.severity.signals[:3])}")
    print(f"       Labels: {r.labels.auto_apply}")
    print()

print(f"\n✓ NON-ESCALATED ({len(non_escalated)} issues):")
for r in non_escalated:
    print(f"  [{r.severity.severity}] {r.issue_id}: {r.title[:50]}")
    print(f"       Type: {r.classification.issue_type}")

print(f"\nEscalation rate: {len(escalated)}/{len(agent.results)} "
      f"({len(escalated)/len(agent.results)*100:.0f}%)")

## 14. Draft Response Preview

In [ ]:
# Show draft responses for a representative sample
sample_ids = [0, 2, 5, 8]  # Bug, question, security, enhancement

for idx in sample_ids:
    r = agent.results[idx]
    print(f"{'='*60}")
    print(f"{r.issue_id}: {r.title[:50]}")
    print(f"Type: {r.classification.issue_type} | "
          f"Severity: {r.severity.severity} | Tone: {r.response.tone}")
    print(f"{'='*60}")
    print(r.response.full_text)
    print()

## 15. Agent Evaluation

In [ ]:
# Expected outcomes for evaluation
EXPECTED = {
    "ISS-001": {"type": IssueType.BUG,             "min_sev": "P0", "escalate": True},
    "ISS-002": {"type": IssueType.FEATURE_REQUEST,  "min_sev": "P3", "escalate": False},
    "ISS-003": {"type": IssueType.QUESTION,          "min_sev": "P3", "escalate": False},
    "ISS-004": {"type": IssueType.PERFORMANCE,       "min_sev": "P1", "escalate": False},
    "ISS-005": {"type": IssueType.DOCUMENTATION,     "min_sev": "P3", "escalate": False},
    "ISS-006": {"type": IssueType.SECURITY,          "min_sev": "P0", "escalate": True},
    "ISS-007": {"type": IssueType.BUG,               "min_sev": "P3", "escalate": False},
    "ISS-008": {"type": IssueType.BUG,               "min_sev": "P2", "escalate": False},
    "ISS-009": {"type": IssueType.FEATURE_REQUEST,   "min_sev": "P3", "escalate": False},
    "ISS-010": {"type": IssueType.PERFORMANCE,       "min_sev": "P1", "escalate": False},
    "ISS-011": {"type": IssueType.BUG,               "min_sev": "P2", "escalate": False},
    "ISS-012": {"type": IssueType.FEATURE_REQUEST,   "min_sev": "P3", "escalate": False},
}

sev_rank = {"P0": 0, "P1": 1, "P2": 2, "P3": 3, "P4": 4}

eval_rows = []
for r in agent.results:
    exp = EXPECTED.get(r.issue_id, {})
    if not exp:
        continue

    type_ok = r.classification.issue_type == exp["type"]
    sev_ok = sev_rank.get(str(r.severity.severity), 4) <= sev_rank.get(exp["min_sev"], 4)
    esc_ok = r.severity.escalate == exp["escalate"]
    has_labels = len(r.labels.labels) > 0
    valid = len(r.validate()) == 0

    eval_rows.append({
        "issue_id": r.issue_id,
        "type_correct": type_ok,
        "severity_ok": sev_ok,
        "escalation_ok": esc_ok,
        "has_labels": has_labels,
        "schema_valid": valid,
        "all_pass": type_ok and sev_ok and esc_ok and has_labels and valid,
    })

eval_df = pd.DataFrame(eval_rows)
print("EVALUATION RESULTS")
print("=" * 80)
for _, row in eval_df.iterrows():
    checks = [
        ("type", row["type_correct"]),
        ("sev", row["severity_ok"]),
        ("esc", row["escalation_ok"]),
        ("lbl", row["has_labels"]),
        ("schema", row["schema_valid"]),
    ]
    detail = "  ".join(f"{n}={'ok' if v else 'FAIL'}" for n, v in checks)
    icon = "pass" if row["all_pass"] else "FAIL"
    print(f"  {'['+icon+']':8s} {row['issue_id']}  {detail}")

col_pass = {col: eval_df[col].mean() for col in
            ["type_correct", "severity_ok", "escalation_ok", "has_labels", "schema_valid"]}
print(f"\nPer-check pass rates:")
for col, rate in col_pass.items():
    print(f"  {col:20s} {rate:.0%}")
print(f"\nOverall pass rate: {eval_df['all_pass'].mean():.0%}")

In [ ]:
# Confusion-style analysis: predicted vs expected types
from collections import Counter

pred_types = [str(r.classification.issue_type) for r in agent.results]
exp_types = [str(EXPECTED.get(r.issue_id, {}).get("type", "?")) for r in agent.results]

# Build confusion data
type_labels = sorted(set(pred_types + exp_types))
confusion = np.zeros((len(type_labels), len(type_labels)), dtype=int)
for p, e in zip(pred_types, exp_types):
    if e == "?":
        continue
    pi = type_labels.index(p)
    ei = type_labels.index(e)
    confusion[ei, pi] += 1

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(confusion, cmap="Blues", aspect="auto")
ax.set_xticks(range(len(type_labels)))
ax.set_yticks(range(len(type_labels)))
ax.set_xticklabels(type_labels, rotation=45, ha="right", fontsize=8)
ax.set_yticklabels(type_labels, fontsize=8)
ax.set_xlabel("Predicted Type")
ax.set_ylabel("Expected Type")
ax.set_title("Classification Confusion Matrix")

for i in range(len(type_labels)):
    for j in range(len(type_labels)):
        if confusion[i, j] > 0:
            ax.text(j, i, str(confusion[i, j]), ha="center", va="center",
                    color="white" if confusion[i, j] > 1 else "black", fontsize=11)

plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()

## 16. Severity Heatmap by Component

In [ ]:
# Build a component x severity heatmap
comp_sev = defaultdict(lambda: defaultdict(int))
for r in agent.results:
    sev = str(r.severity.severity)
    for cm in r.labels.component_matches:
        comp_sev[str(cm.component)][sev] += 1

if comp_sev:
    comps = sorted(comp_sev.keys())
    sevs = ["P0", "P1", "P2", "P3", "P4"]

    matrix = np.zeros((len(comps), len(sevs)), dtype=int)
    for i, comp in enumerate(comps):
        for j, sev in enumerate(sevs):
            matrix[i, j] = comp_sev[comp].get(sev, 0)

    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(matrix, cmap="YlOrRd", aspect="auto")
    ax.set_xticks(range(len(sevs)))
    ax.set_yticks(range(len(comps)))
    ax.set_xticklabels(sevs)
    ax.set_yticklabels(comps, fontsize=9)
    ax.set_xlabel("Severity")
    ax.set_ylabel("Component")
    ax.set_title("Issue Severity by Component")

    for i in range(len(comps)):
        for j in range(len(sevs)):
            if matrix[i, j] > 0:
                ax.text(j, i, str(matrix[i, j]), ha="center", va="center",
                        fontsize=10, color="black")

    plt.colorbar(im, ax=ax, shrink=0.8, label="Issue Count")
    plt.tight_layout()
    plt.show()

## 17. Save Experiment Log

In [ ]:
log = {
    "timestamp": datetime.now().isoformat(),
    "task": "issue_triage_agent",
    "issues_triaged": len(agent.results),
    "type_distribution": dict(Counter(str(r.classification.issue_type) for r in agent.results)),
    "severity_distribution": dict(Counter(str(r.severity.severity) for r in agent.results)),
    "escalation_count": sum(1 for r in agent.results if r.severity.escalate),
    "avg_classification_conf": np.mean([r.classification.confidence for r in agent.results]).item(),
    "avg_severity_conf": np.mean([r.severity.confidence for r in agent.results]).item(),
    "all_results_valid": all(len(r.validate()) == 0 for r in agent.results),
    "structured_output_fields": list(TriageResult.__dataclass_fields__.keys()),
    "enums_used": {
        "IssueType": [e.value for e in IssueType],
        "Component": [e.value for e in Component],
        "Severity": [e.value for e in Severity],
    },
}

log_path = ARTIFACT_DIR / "issue_triage_agent_log.json"
log_path.write_text(json.dumps(log, indent=2, default=str), encoding="utf-8")
print(f"Saved: {log_path}")

# Also save all structured triage results
results_path = ARTIFACT_DIR / "triage_results.json"
results_data = [r.to_dict() for r in agent.results]
results_path.write_text(json.dumps(results_data, indent=2, default=str), encoding="utf-8")
print(f"Saved: {results_path}")

print(f"\nFinal stats:")
print(f"  Issues triaged:      {log['issues_triaged']}")
print(f"  Escalated:           {log['escalation_count']}")
print(f"  Avg type confidence: {log['avg_classification_conf']:.2f}")
print(f"  Avg sev confidence:  {log['avg_severity_conf']:.2f}")
print(f"  All outputs valid:   {log['all_results_valid']}")

## 18. Key Takeaways

### What We Built
- An **issue triage agent** with four stages: classify → label → severity → respond
- **7 issue types** (bug, feature, question, docs, performance, security, enhancement)
- **11 component areas** for automatic label suggestion
- **5-level severity scale** (P0–P4) with signal-based estimation
- **Structured outputs** at every stage — typed dataclasses with enum constraints and validation
- **Draft responses** with tone-appropriate templates and severity-based SLAs
- Full **evaluation suite** with expected outcomes and confusion matrix

### Structured Output Principles
1. **Use Enums** for categorical fields — prevents invalid values
2. **Use dataclasses** for typed, serializable records
3. **Include confidence scores** — express uncertainty numerically
4. **Include reasoning** — every decision must be explainable
5. **Validate at construction** — catch errors before they propagate
6. **Separate auto-apply from needs-review** — route by confidence
7. **Serialize to JSON** — for logging, APIs, and downstream systems

### Production Path
| Enhancement | Purpose |
|---|---|
| **LLM classification** | Replace keyword rules with GPT-4o / Claude |
| **OpenAI Structured Outputs** | `response_format=TriageOutput` for guaranteed schema |
| **Pydantic models** | Richer validation than dataclasses |
| **Feedback loop** | Human corrections improve future triage |
| **SLA integration** | Route P0/P1 to PagerDuty, P3/P4 to backlog |
| **Duplicate detection** | Find similar existing issues before triaging |
| **Multi-language** | Support non-English issue reports |
| **Embedding-based classification** | Sentence embeddings + nearest-neighbor for edge cases |
